# PACT — 02: analysis, figures and LaTeX tables

Reads the JSON run records produced by notebook 01 and writes

* `paper/results/tab_fidelity.tex` → Table 2
* `paper/results/tab_explanation.tex` → Table 3
* `paper/results/tab_certificate.tex` → Table 4
* `paper/results/tab_hparams.tex` → Table 5
* `paper/results/tab_ablation_rho.tex` → Table 6
* `figures/frontier.pdf`, `figures/agreement_vs_eta.pdf`

**No number in the report is typed by hand.** `paper/main.tex` `\input`s these
files, and renders a visible placeholder box wherever one is missing — so a
build with boxes in it is a build with no experiments behind it.

In [ ]:
#@title Setup — clone the repo and install dependencies { display-mode: "form" }
REPO_URL  = "https://github.com/<user>/pact-hdmr-kd.git"  #@param {type:"string"}
USE_DRIVE = True   #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/pact_runs"  #@param {type:"string"}

import os, subprocess, sys, pathlib

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not pathlib.Path("/content/pact-hdmr-kd").exists():
        # If you uploaded the repo as a zip instead, unzip it to /content and
        # the clone is skipped automatically.
        try:
            subprocess.run(["git", "clone", "-q", REPO_URL,
                            "/content/pact-hdmr-kd"], check=True)
        except Exception as e:
            print("clone failed:", e)
            print("Upload the repo zip to /content and unzip it, then re-run.")
    os.chdir("/content/pact-hdmr-kd")

    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "interpret-core==0.6.*", "shap>=0.44", "openml"], check=False)

    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        pathlib.Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

sys.path.insert(0, os.getcwd())

import numpy as np, torch
print("cwd       :", os.getcwd())
print("torch     :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu       :", torch.cuda.get_device_name(0))

In [ ]:
from pact.experiment import load_runs
from pact.config import RunConfig

OUT_DIR = DRIVE_DIR if (IN_COLAB and USE_DRIVE) else "runs"
runs = load_runs(OUT_DIR)
assert runs, f"no run records in {OUT_DIR} — run notebook 01 first"

RHO_MAIN = 0.5      #@param {type:"number"}
TEACHER  = "mlp"    #@param ["mlp", "gbm"]
print(f"{len(runs)} records loaded from {OUT_DIR}")

## 1. The headline question (OP1): does label agreement predict explanation error?

If the two were tightly coupled, the prediction-level fidelity numbers reported
throughout the transparent-distillation literature would already control
explanation error and the certificate would be redundant. The rank correlation
below is the test.

In [ ]:
import numpy as np
from scipy.stats import spearmanr

agree, eta, labels = [], [], []
for r in runs:
    for sname, s in r["students"].items():
        fm = s.get("fidelity_mu")
        if fm and np.isfinite(fm["eta"]) and np.isfinite(fm["top1_agreement"]):
            agree.append(fm["top1_agreement"]); eta.append(fm["eta"])
            labels.append((r["dataset"], sname))

rho_s = spearmanr(agree, eta).statistic
print(f"n = {len(agree)} (dataset x student x seed)")
print(f"Spearman(top-1 agreement, eta_mu) = {rho_s:.3f}")
print()
print("A strongly negative value means agreement does track the certified")
print("residual; a weak one means the literature's fidelity numbers do not")
print("control explanation error. Either way, report what you measured.")

## 2. Certificate audit

`holds` must be 1.00 everywhere. `tightness` = measured / bound is the price of
a worst-case argument — a very small number means the bound is loose, which is
a finding to report honestly rather than hide.

In [ ]:
import numpy as np
rows = []
for r in runs:
    c = r["students"]["gas_pce"].get("certificate")
    if c:
        rows.append((r["dataset"], r["rho"], c["eta_mu"], c["bound_delta"],
                     c["measured_l1"], c["tightness"], c["bound_holds"],
                     c.get("vacuous")))

import collections
agg = collections.defaultdict(list)
for d, rho, e, b, m, t, h, v in rows:
    agg[(d, rho)].append((e, b, m, t, h, v))

print(f"{'dataset':9s} {'rho':>5s} {'eta_mu':>8s} {'Delta':>8s} "
      f"{'measured':>9s} {'tight':>7s} {'holds':>6s} {'vacuous':>8s}")
for (d, rho), vals in sorted(agg.items()):
    a = np.array([[x[0], x[1], x[2], x[3]] for x in vals], dtype=float)
    a = np.where(np.isfinite(a), a, np.nan)
    holds = np.mean([x[4] for x in vals])
    vac = np.mean([bool(x[5]) for x in vals])
    print(f"{d:9s} {rho:5g} {np.nanmean(a[:,0]):8.3f} {np.nanmean(a[:,1]):8.3f} "
          f"{np.nanmean(a[:,2]):9.3f} {np.nanmean(a[:,3]):7.3f} "
          f"{holds:6.2f} {vac:8.2f}")

## 3. Failure-mode diagnostics (§6 of the report)

**(ii) Off-manifold divergence.** The `eta_mu − eta_nu` gap at ρ = 0 is
Proposition 3 made visible: a student that looks faithful on the data can be
far from the teacher on the measure its own explanation is defined under.

In [ ]:
import collections, numpy as np
gap = collections.defaultdict(list)
for r in runs:
    g = r["students"]["gas_pce"]
    gap[(r["dataset"], r["rho"])].append(
        g["fidelity_mu"]["eta"] - g["fidelity_nu"]["eta"])

print(f"{'dataset':9s} {'rho':>5s} {'eta_mu - eta_nu':>17s}")
for k, v in sorted(gap.items()):
    v = np.array(v, dtype=float)
    print(f"{k[0]:9s} {k[1]:5g} {np.nanmean(v):17.4f}")

**(iv) Dependence-induced attribution splitting.** When two features are
near-duplicates the teacher's variance is shared between them arbitrarily, so
individual total effects wobble across seeds while their *sum* is stable.
Large per-feature variance with small cluster variance is diagnostic of this
mode rather than of a bad student.

In [ ]:
import numpy as np, collections

def seed_stability(runs, dataset, rho, student="gas_pce"):
    te = [np.array(r["students"][student]["total_effects"])
          for r in runs if r["dataset"] == dataset
          and abs(r["rho"] - rho) < 1e-9
          and "total_effects" in r["students"].get(student, {})]
    if len(te) < 2:
        return None
    T = np.vstack(te)
    return {"n_seeds": T.shape[0],
            "mean_per_feature_sd": float(T.std(axis=0, ddof=1).mean()),
            "sd_of_top5_mass": float(
                np.std([np.sort(t)[::-1][:5].sum() for t in T], ddof=1))}

for d in ("bank", "credit", "gas", "musk"):
    s = seed_stability(runs, d, RHO_MAIN)
    if s:
        print(f"{d:9s} seeds={s['n_seeds']:2d}  "
              f"mean per-feature sd={s['mean_per_feature_sd']:.4f}  "
              f"sd of top-5 mass={s['sd_of_top5_mass']:.4f}")

**(i) Truncation floor.** If `eta_mu` stops improving as λ decreases, the limit
is truncation (order ≤ 2, top-*m* screening), not regularisation. The frontier
makes this visible as a horizontal asymptote.

In [ ]:
import numpy as np, collections
for d in ("bank", "credit", "gas", "musk"):
    curves = [r["pce_frontier"] for r in runs
              if r["dataset"] == d and abs(r["rho"] - RHO_MAIN) < 1e-9
              and r.get("pce_frontier")]
    if not curves:
        continue
    n = min(len(c) for c in curves)
    eta = np.median([[p["eta_mu"] for p in c[:n]] for c in curves], axis=0)
    comp = np.median([[p["active_components"] for p in c[:n]] for c in curves], axis=0)
    print(f"{d:9s} eta_mu: {eta[0]:.3f} (|U|={comp[0]:.0f}) -> "
          f"{eta[-1]:.3f} (|U|={comp[-1]:.0f})   "
          f"floor gap last two: {abs(eta[-1]-eta[-2]):.4f}")

## 4. Emit the LaTeX tables and the figures

In [ ]:
from pact import tables, figures
import pathlib

pathlib.Path("paper/results").mkdir(parents=True, exist_ok=True)
pathlib.Path("figures").mkdir(parents=True, exist_ok=True)

written = tables.emit_all(runs, "paper/results", rho=RHO_MAIN, teacher=TEACHER)
figs = figures.emit_all(runs, "figures", rho=RHO_MAIN, teacher=TEACHER)

for w in written:
    print("table :", w)
for f in figs:
    print("figure:", f)

print("\n--- tab_certificate.tex ---")
print(pathlib.Path("paper/results/tab_certificate.tex").read_text())

In [ ]:
# preview the figures inline
from IPython.display import display
import matplotlib.pyplot as plt, matplotlib.image as mpimg, subprocess, pathlib

for name in ("frontier", "agreement_vs_eta"):
    pdf = f"figures/{name}.pdf"
    if pathlib.Path(pdf).exists():
        try:
            subprocess.run(["pdftoppm", "-png", "-r", "110", pdf,
                            f"figures/{name}"], check=True)
            png = sorted(pathlib.Path("figures").glob(f"{name}-*.png"))[0]
            plt.figure(figsize=(6, 4.5)); plt.axis("off")
            plt.imshow(mpimg.imread(png)); plt.show()
        except Exception as e:
            print(f"(preview unavailable for {name}: {e})")

## 5. Package the artefacts for the report

Download `pact_artifacts.zip`, unzip it into `paper/` in the repo, and
recompile:

```bash
unzip -o pact_artifacts.zip -d paper/
cd paper && pdflatex main && bibtex main && pdflatex main && pdflatex main
```

In [ ]:
import pathlib, zipfile
# a tight archive with just what the paper needs:
with zipfile.ZipFile("pact_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in pathlib.Path("paper/results").glob("*.tex"):
        z.write(p, f"results/{p.name}")
    for p in pathlib.Path("figures").glob("*.pdf"):
        z.write(p, f"figures/{p.name}")
print("wrote pact_artifacts.zip:",
      zipfile.ZipFile("pact_artifacts.zip").namelist())

if IN_COLAB:
    from google.colab import files
    files.download("pact_artifacts.zip")

---
### Before you submit

- [ ] `holds` is 1.00 for every dataset in §2 above
- [ ] no placeholder boxes left in the compiled PDF
- [ ] report ≤ 8 pages excluding references; Part A ≤ 3 pages
- [ ] the AI-use statement at the end of `main.tex` matches what you actually did